# MetaPhlAn4 tutorial
MetaPhlAn relies on unique clade-specific marker genes that were identified from ~ 1M microbial genomes from bacteria, archaea, viruses and eukaryotes (further details)

*   ~236,600 reference genomes
*   ~771,500 metagenomic assembled genomes

Illustration of clades [here](https://drive.google.com/file/d/1Ln_-JTfAG6fNuSl9RpLKjUQeynL229OK/view?usp=drive_link)

# Workflow:

**Input**:


*   QC'ed reads
*   Clade-specific marker gene database


**Output**:


*   Marker genes mapping file [.sam]
*   Microbial profile table [.txt]





# Setup MetaPhlAn

In [11]:
# check if the installation is successful
import os,sys
! which bowtie2
! which metaphlan

/opt/conda/bin/bowtie2
/opt/conda/bin/metaphlan


# Input data:

In [16]:
# mgx reads folder
root_dir = "/biodata/resources/MetaPhlan_intro"
mgx_reads_dir = os.path.join(root_dir,"mgx_reads")
! ls -lh {mgx_reads_dir}
sample_id="PSMB4MBK_subset"
metaphlan_profile_dir =  os.path.join(root_dir,"reference_based_metagenomic_analysis","profile")
! mkdir -p {metaphlan_profile_dir}
! zcat {mgx_reads_dir}/{sample_id}_R1.fastq.gz | head -n 4 2>/dev/null

total 980K
-rw-rw-rw- 1 root root  15K Sep 30  2025 PSMB4MBK.log
-rw-rw-rw- 1 root root 482K Sep 30  2025 PSMB4MBK_subset_R1.fastq.gz
-rw-rw-rw- 1 root root 478K Sep 30  2025 PSMB4MBK_subset_R2.fastq.gz
@CAVL1ANXX170419:1:1101:10000:97843/1
CGAAGAGCCTTCGCGTGATCGGCATCAGGAACTTGGGATTGGCGGGATCG
+
<BBBBFFFFFFFFFFFFFFFFFFBBFFFFFFFFFFFBFFFFFFFFFFFFF

gzip: stdout: Broken pipe


## Count the number of lines and the number of reads in the fastq file

In [17]:
! zcat {mgx_reads_dir}/{sample_id}_R1.fastq.gz | wc -l

40000


In [18]:
34497952/4

8624488.0

## Reminder: fastq format

Four lines per record [per read]


1.   Read identifier [starts with @, ends with /1 or /2]
2.   Nucleotide sequence
3.   Place holder / separater [+]
4.   Phred score [quality score for each nucleotide position]

Phred score table [here](https://drive.google.com/file/d/1poJ6joQKbhJO9btjvV5xEvRr2_0uGAxs/view?usp=drive_link)

## The Phred score encoding system:

image [here](https://drive.google.com/file/d/1WNVk2h8AFKKcXyxSA2h0dyEAcRlVkKzL/view?usp=drive_link)

# Running MetaPhlAn

In [19]:
# metaphlan database
metaphlan_db_dir = os.path.join(root_dir,"reference_based_metagenomic_analysis","Metaphlan4_DB")
! ls -lsh {metaphlan_db_dir}

total 41M
4.0K -rw-rw-rw- 1 root root  26 Sep 30  2025 mpa_latest
4.0K -rw-rw-rw- 1 root root  26 Sep 30  2025 mpa_previous
 12M -rw-rw-rw- 1 root root 12M Sep 30  2025 mpa_vOct22_CHOCOPhlAnSGB_202403.3.bt2l
4.0K -rw-rw-rw- 1 root root  64 Sep 30  2025 mpa_vOct22_CHOCOPhlAnSGB_202403.md5
 29M -rw-rw-rw- 1 root root 29M Sep 30  2025 mpa_vOct22_CHOCOPhlAnSGB_202403.pkl
4.0K -rw-rw-rw- 1 root root  50 Sep 30  2025 README.txt


## Command format:

In [20]:
! metaphlan --help

usage: metaphlan --input_type {fastq,fasta,bowtie2out,sam} [--force]
                 [--bowtie2db METAPHLAN_BOWTIE2_DB] [-x INDEX]
                 [--bt2_ps BowTie2 presets] [--bowtie2_exe BOWTIE2_EXE]
                 [--bowtie2_build BOWTIE2_BUILD] [--bowtie2out FILE_NAME]
                 [--min_mapq_val MIN_MAPQ_VAL] [--no_map] [--tmp_dir]
                 [--tax_lev TAXONOMIC_LEVEL] [--min_cu_len]
                 [--min_alignment_len] [--add_viruses] [--ignore_eukaryotes]
                 [--ignore_bacteria] [--ignore_archaea] [--ignore_ksgbs]
                 [--ignore_usgbs] [--stat_q] [--perc_nonzero]
                 [--ignore_markers IGNORE_MARKERS] [--avoid_disqm] [--stat]
                 [-t ANALYSIS TYPE] [--nreads NUMBER_OF_READS]
                 [--pres_th PRESENCE_THRESHOLD] [--clade] [--min_ab]
                 [--profile_vsc] [--vsc_out VSC_OUT]
                 [--vsc_breadth VSC_BREADTH] [-o output file]
                 [--sample_id_key name] [--use_group_repr

Relavent parameters
*   rel_ab_w_read_stats: profiling relative abundances and estimating the number of reads coming from each clade.
*   nproc: The number of CPUs to use for parallelizing the mapping step [default 4]




In [21]:
# check input fastq files
! ls -lh {mgx_reads_dir}/{sample_id}_R*.fastq.gz
# create the output folder
! mkdir -p {metaphlan_profile_dir}

-rw-rw-rw- 1 root root 482K Sep 30  2025 /biodata/resources/MetaPhlan_intro/mgx_reads/PSMB4MBK_subset_R1.fastq.gz
-rw-rw-rw- 1 root root 478K Sep 30  2025 /biodata/resources/MetaPhlan_intro/mgx_reads/PSMB4MBK_subset_R2.fastq.gz


In [22]:
!echo {metaphlan_db_dir}

/biodata/resources/MetaPhlan_intro/reference_based_metagenomic_analysis/Metaphlan4_DB


This is the command for running metaphlan. As you would usually execute this on a server with more compute power, we provide the pre-computed output below.

In [204]:
#DO NOT EXECUTE
#! zcat {mgx_reads_dir}/{sample_id}_R*.fastq.gz | metaphlan --bowtie2db {metaphlan_db_dir} -x mpa_v31_CHOCOPhlAn_201901 -t rel_ab_w_read_stats --nproc 8 --input_type fastq > {metaphlan_profile_dir}/{sample_id}.profile.txt;

## Check the output:

In [23]:
! ls -lh {metaphlan_profile_dir}/{sample_id}.profile.txt;
! head {metaphlan_profile_dir}/{sample_id}.profile.txt

-rw-rw-rw- 1 root root 2.0K Aug 21 15:44 /biodata/resources/MetaPhlan_intro/reference_based_metagenomic_analysis/profile/PSMB4MBK_subset.profile.txt
#mpa_vOct22_CHOCOPhlAnSGB_202403
#/nfs/home/tingting/mambaforge/bin/metaphlan PSMB4MBK_subset.fa --bowtie2out PSMB4MBK_subset.bowtie2.bz2 --nproc 40 --input_type fasta -o PSMB4MBK_subset.profile.txt --index mpa_vOct22_CHOCOPhlAnSGB_202403 --bowtie2db /nfs/home/tingting/src/metaphlan4.1.1_databasevMar24
#17051 reads processed
#SampleID	Metaphlan_Analysis
#clade_name	NCBI_tax_id	relative_abundance	additional_species
k__Bacteria	2	100.0	
k__Bacteria|p__Firmicutes	2|1239	100.0	
k__Bacteria|p__Firmicutes|c__Negativicutes	2|1239|909932	54.58217	
k__Bacteria|p__Firmicutes|c__Clostridia	2|1239|186801	45.41783	
k__Bacteria|p__Firmicutes|c__Negativicutes|o__Veillonellales	2|1239|909932|1843489	54.58217	


In [24]:
!grep --help

Usage: grep [OPTION]... PATTERNS [FILE]...
Search for PATTERNS in each FILE.
Example: grep -i 'hello world' menu.h main.c
PATTERNS can contain multiple patterns separated by newlines.

Pattern selection and interpretation:
  -E, --extended-regexp     PATTERNS are extended regular expressions
  -F, --fixed-strings       PATTERNS are strings
  -G, --basic-regexp        PATTERNS are basic regular expressions
  -P, --perl-regexp         PATTERNS are Perl regular expressions
  -e, --regexp=PATTERNS     use PATTERNS for matching
  -f, --file=FILE           take PATTERNS from FILE
  -i, --ignore-case         ignore case distinctions in patterns and data
      --no-ignore-case      do not ignore case distinctions (default)
  -w, --word-regexp         match only whole words
  -x, --line-regexp         match only whole lines
  -z, --null-data           a data line ends in 0 byte, not newline

Miscellaneous:
  -s, --no-messages         suppress error messages
  -v, --invert-match        select no

In [25]:
# An example of bacterial lineage
! grep s__ -m1 {metaphlan_profile_dir}/{sample_id}.profile.txt #| cut -f1 | sed 's/|/\n/g'

k__Bacteria|p__Firmicutes|c__Negativicutes|o__Veillonellales|f__Veillonellaceae|g__Dialister|s__Dialister_invisus	2|1239|909932|1843489|31977|39948|218538	54.58217	


## Merging tables from multiple samples

In [26]:
! ls -lh {metaphlan_profile_dir}/*.profile4.txt

-rw-rw-rw- 1 root root 46K Aug 21 16:25 /biodata/resources/MetaPhlan_intro/reference_based_metagenomic_analysis/profile/CSM9X233.profile4.txt
-rw-rw-rw- 1 root root 16K Aug 21 16:25 /biodata/resources/MetaPhlan_intro/reference_based_metagenomic_analysis/profile/HSMA33OT.profile4.txt


In [27]:
! merge_metaphlan_tables.py {metaphlan_profile_dir}/*profile4.txt > {metaphlan_profile_dir}/merged_abundance_table.txt

In [28]:
# check the output
! head {metaphlan_profile_dir}/merged_abundance_table.txt


#mpa_vOct22_CHOCOPhlAnSGB_202403
clade_name	CSM9X233.profile4	HSMA33OT.profile4
k__Bacteria	100.0	100.0
k__Bacteria|p__Firmicutes	61.27325	18.32166
k__Bacteria|p__Bacteroidetes	32.77716	80.34634
k__Bacteria|p__Actinobacteria	4.86351	0.0
k__Bacteria|p__Proteobacteria	1.08607	1.332
k__Bacteria|p__Firmicutes|c__Clostridia	55.89572	18.21812
k__Bacteria|p__Bacteroidetes|c__Bacteroidia	32.77716	80.34634
k__Bacteria|p__Firmicutes|c__Negativicutes	4.40161	0.10354


In [33]:
# check species level abundance
#--line-buffered option may reduce buffering, so that you would not see the error "grep: write error: Broken pipe"
! grep --line-buffered "s__" {metaphlan_profile_dir}/merged_abundance_table.txt | head -n 10

k__Bacteria|p__Firmicutes|c__Clostridia|o__Eubacteriales|f__Oscillospiraceae|g__Faecalibacterium|s__Faecalibacterium_prausnitzii	27.10231	8.96372
k__Bacteria|p__Bacteroidetes|c__Bacteroidia|o__Bacteroidales|f__Bacteroidaceae|g__Phocaeicola|s__Phocaeicola_coprocola	18.10004	0.0
k__Bacteria|p__Bacteroidetes|c__Bacteroidia|o__Bacteroidales|f__Bacteroidaceae|g__Bacteroides|s__Bacteroides_stercoris	7.6534	19.84925
k__Bacteria|p__Firmicutes|c__Clostridia|o__Eubacteriales|f__Oscillospiraceae|g__Faecalibacterium|s__Faecalibacterium_SGB15346	7.36332	0.0
k__Bacteria|p__Firmicutes|c__Negativicutes|o__Veillonellales|f__Veillonellaceae|g__Dialister|s__Dialister_invisus	4.06708	0.10354
k__Bacteria|p__Bacteroidetes|c__Bacteroidia|o__Bacteroidales|f__Bacteroidaceae|g__Phocaeicola|s__Phocaeicola_vulgatus	3.69917	38.66308
k__Bacteria|p__Firmicutes|c__Clostridia|o__Eubacteriales|f__Lachnospiraceae|g__Lachnospiraceae_unclassified|s__Eubacterium_rectale	3.67402	1.54263
k__Bacteria|p__Firmicutes|c__Clostrid

In [36]:
! grep "t__" {metaphlan_profile_dir}/merged_abundance_table.txt|head -n 10

k__Bacteria|p__Bacteroidetes|c__Bacteroidia|o__Bacteroidales|f__Bacteroidaceae|g__Phocaeicola|s__Phocaeicola_coprocola|t__SGB1891	18.10004	0.0
k__Bacteria|p__Firmicutes|c__Clostridia|o__Eubacteriales|f__Oscillospiraceae|g__Faecalibacterium|s__Faecalibacterium_prausnitzii|t__SGB15342	16.71421	0.0
k__Bacteria|p__Bacteroidetes|c__Bacteroidia|o__Bacteroidales|f__Bacteroidaceae|g__Bacteroides|s__Bacteroides_stercoris|t__SGB1830	7.6534	19.84925
k__Bacteria|p__Firmicutes|c__Clostridia|o__Eubacteriales|f__Oscillospiraceae|g__Faecalibacterium|s__Faecalibacterium_SGB15346|t__SGB15346	7.36332	0.0
k__Bacteria|p__Firmicutes|c__Clostridia|o__Eubacteriales|f__Oscillospiraceae|g__Faecalibacterium|s__Faecalibacterium_prausnitzii|t__SGB15332	5.97784	0.0
k__Bacteria|p__Firmicutes|c__Clostridia|o__Eubacteriales|f__Oscillospiraceae|g__Faecalibacterium|s__Faecalibacterium_prausnitzii|t__SGB15318	4.38789	0.0
k__Bacteria|p__Firmicutes|c__Negativicutes|o__Veillonellales|f__Veillonellaceae|g__Dialister|s__Diali

In [30]:
! grep "p__" {metaphlan_profile_dir}/merged_abundance_table.txt|head -n 10

k__Bacteria|p__Firmicutes	61.27325	18.32166
k__Bacteria|p__Bacteroidetes	32.77716	80.34634
k__Bacteria|p__Actinobacteria	4.86351	0.0
k__Bacteria|p__Proteobacteria	1.08607	1.332
k__Bacteria|p__Firmicutes|c__Clostridia	55.89572	18.21812
k__Bacteria|p__Bacteroidetes|c__Bacteroidia	32.77716	80.34634
k__Bacteria|p__Firmicutes|c__Negativicutes	4.40161	0.10354
k__Bacteria|p__Actinobacteria|c__Actinomycetia	3.48083	0.0
k__Bacteria|p__Actinobacteria|c__Coriobacteriia	1.38269	0.0
k__Bacteria|p__Proteobacteria|c__Betaproteobacteria	1.08105	1.22938
grep: write error: Broken pipe


In [34]:
# check phylym level abundance
! grep "p__" {metaphlan_profile_dir}/merged_abundance_table.txt | grep -v "c__"


k__Bacteria|p__Firmicutes	61.27325	18.32166
k__Bacteria|p__Bacteroidetes	32.77716	80.34634
k__Bacteria|p__Actinobacteria	4.86351	0.0
k__Bacteria|p__Proteobacteria	1.08607	1.332


In [35]:
# check order Bacteroidales at family level abundance
! grep "o__Bacteroidales" {metaphlan_profile_dir}/merged_abundance_table.txt | grep -v "g__"

k__Bacteria|p__Bacteroidetes|c__Bacteroidia|o__Bacteroidales	32.77716	80.34634
k__Bacteria|p__Bacteroidetes|c__Bacteroidia|o__Bacteroidales|f__Bacteroidaceae	32.03392	74.51716
k__Bacteria|p__Bacteroidetes|c__Bacteroidia|o__Bacteroidales|f__Tannerellaceae	0.41326	4.94177
k__Bacteria|p__Bacteroidetes|c__Bacteroidia|o__Bacteroidales|f__Odoribacteraceae	0.22568	0.0
k__Bacteria|p__Bacteroidetes|c__Bacteroidia|o__Bacteroidales|f__Rikenellaceae	0.10431	0.88741


## Task: how many species were identified?

Hint:


*   **grep** species-level records
*   use **wc** command to count the number of records

